# Laya fine-tune for tickertape (Kaggle)

Settings: **Accelerator** GPU T4 x2 (one GPU is used) or P100, **Internet** On, **Secrets**: add `HF_TOKEN` (a Hugging Face *write* token) and attach it to this notebook, **Input**: a private Kaggle dataset containing `train.jsonl`, `val.jsonl` and `finetune_laya.py`.

Never add `gold_set.jsonl` to the dataset: it is the held-out evaluation set.

The training labels were written by Claude Code (an LLM), not by humans.

In [ ]:
HF_REPO = "YOUR-HF-USERNAME/tickertape-laya"   # private repo, created if missing
EPOCHS = 4

# Same versions as the cluster image. If pip complains about the transformers pin, remove it and keep laya==0.3.24.
!pip install -q laya==0.3.24 transformers==5.18.0
!nvidia-smi -L

In [ ]:
import glob, os
from kaggle_secrets import UserSecretsClient

found = {n: glob.glob(f"/kaggle/input/**/{n}", recursive=True) for n in ("train.jsonl", "val.jsonl", "finetune_laya.py")}
print(found)
assert all(len(v) == 1 for v in found.values()), "expected exactly one of each file in the attached dataset"
assert not glob.glob("/kaggle/input/**/gold_set.jsonl", recursive=True), "remove gold_set.jsonl from the dataset"
TRAIN, VAL, SCRIPT = found["train.jsonl"][0], found["val.jsonl"][0], found["finetune_laya.py"][0]
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # never printed

In [ ]:
# Trains, keeps the best epoch by validation loss, fits calibration temperatures on val, reloads the saved
# checkpoint through laya.load to verify it, then pushes to the private Hub repo.
!python {SCRIPT} --train {TRAIN} --val {VAL} --output-dir /kaggle/working/laya_finetuned --epochs {EPOCHS} --hf-repo {HF_REPO}

In [ ]:
# Copy these numbers back into the chat: per-epoch validation loss, the temperatures, and the pushed commit.
import json
log = json.load(open("/kaggle/working/laya_finetuned/training_log.json"))
print("best epoch", log["best_epoch"], "| temperatures", log["temperature_by_options"], log["temperature"])
for e in log["epochs"]:
    print(e["epoch"], round(e["train_loss"], 4), {k: round(v["ce"], 3) for k, v in e["val"].items()})
from huggingface_hub import HfApi
print("repo head commit:", HfApi().list_repo_commits(HF_REPO)[0].commit_id)